# 📓 Notebook 01 — Data Exploration & Pipeline

**LUMINA Project** · *Real-world data: SEC EDGAR + DocVQA + PubLayNet*

This notebook:
1. Fetches real 10-K filings from SEC EDGAR API (no authentication needed)
2. Explores document structure: length distributions, entity density, chunk quality
3. Builds a reproducible data pipeline saved to `data/`
4. Computes corpus-level statistics for the project report

---

In [ ]:
import sys, os, json, time, re
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import requests
from collections import Counter
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

print('Setup complete ✓')

## 1. Fetch Real 10-K Filings from SEC EDGAR

In [ ]:
# SEC EDGAR full-text search API — completely free, no auth required
SEC_HEADERS = {'User-Agent': 'LUMINA-Research lumina@research.edu'}

# Company CIK numbers (SEC identifiers)
COMPANIES = {
    'Apple':     '0000320193',
    'Microsoft': '0000789019',
    'Google':    '0001652044',
    'Amazon':    '0001018724',
    'Tesla':     '0001318605',
}

def fetch_recent_10k_url(cik: str) -> str:
    """Get URL of most recent 10-K filing for a company."""
    url = f'https://data.sec.gov/submissions/CIK{cik}.json'
    try:
        resp = requests.get(url, headers=SEC_HEADERS, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        filings = data.get('filings', {}).get('recent', {})
        forms = filings.get('form', [])
        acc_nums = filings.get('accessionNumber', [])
        doc_names = filings.get('primaryDocument', [])
        for i, form in enumerate(forms):
            if form == '10-K':
                acc = acc_nums[i].replace('-', '')
                doc = doc_names[i]
                return f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc}/{doc}'
    except Exception as e:
        print(f'  SEC API error for CIK {cik}: {e}')
    return None

def fetch_filing_text(url: str, max_chars: int = 50000) -> str:
    """Download and clean HTML/text from a filing URL."""
    try:
        resp = requests.get(url, headers=SEC_HEADERS, timeout=15)
        resp.raise_for_status()
        text = resp.text
        # Strip HTML tags
        text = re.sub(r'<[^>]+>', ' ', text)
        # Normalise whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        return text[:max_chars]
    except Exception as e:
        return f'[Fetch failed: {e}]'

# Fetch filings
corpus = []
for company, cik in tqdm(COMPANIES.items(), desc='Fetching SEC filings'):
    url = fetch_recent_10k_url(cik)
    if url:
        text = fetch_filing_text(url)
        corpus.append({'company': company, 'cik': cik, 'url': url, 'text': text})
        time.sleep(0.5)  # be respectful of SEC rate limits
    else:
        print(f'  Could not find 10-K for {company}')

# Fallback: use embedded sample data if API unavailable
if len(corpus) == 0:
    print('SEC API unavailable — using embedded sample data')
    corpus = [
        {'company': 'Apple', 'cik': '0000320193', 'url': 'embedded', 'text': 'Apple Inc. reported total net revenues of $394.3 billion for fiscal year 2022. iPhone revenue was $205.5 billion. Services revenue reached $78.1 billion, a record high. Net income was $99.8 billion. The Board approved a $90 billion share repurchase programme.'},
        {'company': 'Microsoft', 'cik': '0000789019', 'url': 'embedded', 'text': 'Microsoft reported revenue of $198.3 billion for fiscal year 2022. Azure and other cloud services revenue increased 28%. Commercial cloud revenue was $91.2 billion, up 32% year-over-year. Operating income was $83.4 billion.'},
        {'company': 'Google', 'cik': '0001652044', 'url': 'embedded', 'text': 'Alphabet Inc. reported revenues of $282.8 billion in 2022. Google advertising revenues were $224.5 billion. Google Cloud revenue was $26.3 billion, up 37%. YouTube advertising revenue was $29.2 billion.'},
    ]

print(f'\nCorpus size: {len(corpus)} filings')
for doc in corpus:
    print(f"  {doc['company']}: {len(doc['text'].split()):,} words")

## 2. Document Structure Analysis

In [ ]:
from agents.orchestrator import chunk_document
from agents.ner_agent import NERAgent

ner = NERAgent()
ner._pipeline = 'fallback'

stats = []
all_chunks = []

for doc in tqdm(corpus, desc='Analysing documents'):
    chunks = chunk_document(doc['text'], chunk_size=256, overlap=32)
    word_counts = [len(c.split()) for c in chunks]
    
    # Entity density per chunk
    entity_counts = []
    for chunk in chunks[:10]:  # sample first 10 for speed
        ents = ner.extract(chunk)
        total = sum(len(v) for v in ents.values())
        entity_counts.append(total)
    
    stats.append({
        'company':         doc['company'],
        'total_words':     len(doc['text'].split()),
        'n_chunks':        len(chunks),
        'mean_chunk_words': np.mean(word_counts),
        'std_chunk_words':  np.std(word_counts),
        'mean_entity_density': np.mean(entity_counts) if entity_counts else 0,
        'total_entities':  sum(entity_counts),
    })
    
    for i, chunk in enumerate(chunks):
        all_chunks.append({'company': doc['company'], 'chunk_id': i, 'text': chunk,
                           'word_count': len(chunk.split())})

stats_df = pd.DataFrame(stats)
chunks_df = pd.DataFrame(all_chunks)

print(stats_df.round(2).to_string(index=False))

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig)
fig.suptitle('LUMINA Corpus — Data Exploration Dashboard', fontsize=14, fontweight='bold')

colors = ['#7F77DD','#1D9E75','#D85A30','#EF9F27','#378ADD']

# 1. Document sizes
ax1 = fig.add_subplot(gs[0, 0])
ax1.barh(stats_df['company'], stats_df['total_words'], color=colors[:len(stats_df)])
ax1.set_title('Document Size (words)')
ax1.set_xlabel('Word count')
ax1.grid(axis='x', alpha=0.3)

# 2. Chunks per document
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(stats_df['company'], stats_df['n_chunks'], color=colors[:len(stats_df)], alpha=0.85)
ax2.set_title('Chunks per Document (size=256)')
ax2.set_ylabel('N chunks')
ax2.grid(axis='y', alpha=0.3)

# 3. Entity density
ax3 = fig.add_subplot(gs[0, 2])
ax3.bar(stats_df['company'], stats_df['mean_entity_density'],
        color=colors[:len(stats_df)], alpha=0.85)
ax3.set_title('Mean Entity Density / Chunk')
ax3.set_ylabel('Entities per chunk')
ax3.grid(axis='y', alpha=0.3)

# 4. Chunk word count distribution
ax4 = fig.add_subplot(gs[1, :])
for i, (company, grp) in enumerate(chunks_df.groupby('company')):
    ax4.hist(grp['word_count'], bins=30, alpha=0.6, label=company,
             color=colors[i % len(colors)], density=True)
ax4.axvline(x=256, color='red', linestyle='--', label='Target chunk size')
ax4.set_title('Chunk Word Count Distribution')
ax4.set_xlabel('Words per chunk'); ax4.set_ylabel('Density')
ax4.legend(); ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Data exploration complete ✓')

## 3. Save Processed Data

In [ ]:
# Save corpus
with open('../data/raw/sec_corpus.json', 'w') as f:
    json.dump(corpus, f, indent=2)

# Save processed chunks
chunks_df.to_csv('../data/processed/chunks.csv', index=False)
stats_df.to_csv('../data/processed/corpus_stats.csv', index=False)

print('Saved:')
print('  data/raw/sec_corpus.json')
print('  data/processed/chunks.csv')
print('  data/processed/corpus_stats.csv')
print(f'\nTotal chunks for training: {len(chunks_df)}')
print(f'Total words across corpus:  {chunks_df["word_count"].sum():,}')